In [1]:
import pandas as pd

df = pd.read_csv("../../output/domain_cath_pred/test_predicted.csv")

In [2]:
df.columns

Index(['domain_id', 'domain_start', 'domain_end', 'class', 'architecture',
       'topology', 'homology', 'protein_sequence', 'real_domain_start',
       'real_domain_end', 'cath', 'protein_length', 'protein_id',
       'protein_chain_id', 'cath_prediction'],
      dtype='object')

In [4]:
import colorsys
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
import os


def get_cath_color(cath_string, plot_seed=0):
    """Generate RGBA color for CATH domain with consistent class colors but varied shades per plot."""
    if cath_string == "NO_DOMAIN_REGION" or pd.isna(cath_string):
        return mcolors.to_rgba('lightgray', alpha=0.7)
    
    class_id = str(cath_string).split('.')[0] if '.' in str(cath_string) else None
    
    # Fixed base hues for CATH classes
    class_hues = {'1': 0.0, '2': 0.66, '3': 0.33, '4': 0.15}
    base_hue = class_hues.get(class_id, 0.5)
    
    # Generate unique variations per plot while keeping class consistency
    label_hash = hash(str(cath_string))
    hue_seed = label_hash + plot_seed * 113
    sat_seed = label_hash + plot_seed * 137
    val_seed = label_hash + plot_seed * 151
    
    hue_shift = (hash(hue_seed) % 1000) / 1000.0 * 0.4 - 0.2  # ±20% shift
    hue = (base_hue + hue_shift) % 1.0
    
    saturation = 0.2 + (hash(sat_seed) % 1000) / 1000.0 * 0.8  # 0.2-1.0
    value = 0.3 + (hash(val_seed) % 1000) / 1000.0 * 0.7      # 0.3-1.0
    
    r, g, b = colorsys.hsv_to_rgb(hue, np.clip(saturation, 0, 1), np.clip(value, 0, 1))
    return (r, g, b, 1.0)


def visualize_cath_predictions(df, output_dir, num_visualizations=5):
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors

    os.makedirs(output_dir, exist_ok=True)
    
    for i, row in df.head(num_visualizations).iterrows():
        domain_id = row['domain_id']
        protein_length = int(row['protein_length'])
        true_cath = row['cath']
        pred_cath = row['cath_prediction']
        
        # Generate plot-specific seed for consistent but varied colors
        plot_seed = hash(str(domain_id))
        
        # Create per-residue labels
        true_labels = ["NO_DOMAIN_REGION"] * protein_length
        pred_labels = ["NO_DOMAIN_REGION"] * protein_length
        
        # Assign TRUE domain labels
        real_start = int(row['real_domain_start'])
        real_end = int(row['real_domain_end'])
        for pos in range(real_start, min(real_end + 1, protein_length)):
            true_labels[pos] = true_cath
        
        # Assign PREDICTED domain labels
        pred_start = int(row['domain_start'])
        pred_end = int(row['domain_end'])
        for pos in range(pred_start, min(pred_end + 1, protein_length)):
            pred_labels[pos] = pred_cath
        
        # Unique labels and color mapping
        all_labels = sorted(list(set(true_labels + pred_labels)))
        label_to_idx = {label: idx for idx, label in enumerate(all_labels)}
        label_colors = {label: get_cath_color(label, plot_seed) for label in all_labels}
        
        cmap_colors = [label_colors[label] for label in all_labels]
        custom_cmap = mcolors.ListedColormap(cmap_colors)
        
        true_values = [label_to_idx[label] for label in true_labels]
        pred_values = [label_to_idx[label] for label in pred_labels]
        
        # Bigger figure for presentation
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
        fig.suptitle(f"Domain {domain_id}",
                     fontsize=24, fontweight='bold')
        
        vmax = len(all_labels) - 1 if len(all_labels) > 1 else 1
        
        # True labels plot
        ax1.imshow(np.array(true_values).reshape(1, -1), cmap=custom_cmap, aspect='auto',
                   extent=[0, protein_length, 0, 1], vmin=0, vmax=vmax)
        ax1.set_yticks([])
        ax1.set_title("True CATH Domains", fontsize=20, fontweight='bold')
        ax1.set_ylabel("True", fontsize=18, fontweight='bold')
        ax1.tick_params(axis='x', labelsize=16)
        
        # Predicted labels plot
        ax2.imshow(np.array(pred_values).reshape(1, -1), cmap=custom_cmap, aspect='auto',
                   extent=[0, protein_length, 0, 1], vmin=0, vmax=vmax)
        ax2.set_yticks([])
        ax2.set_title("Predicted CATH Domains", fontsize=20, fontweight='bold')
        ax2.set_xlabel("Residue Index", fontsize=18, fontweight='bold')
        ax2.set_ylabel("Predicted", fontsize=18, fontweight='bold')
        ax2.tick_params(axis='x', labelsize=16)
        
        # Bigger legend
        handles = [plt.Rectangle((0, 0), 1, 1, color=label_colors[label]) for label in all_labels]
        ax2.legend(handles, all_labels, loc='upper center', bbox_to_anchor=(0.5, -0.3),
                   fancybox=True, shadow=True, ncol=3, fontsize=16, frameon=True)
        
        plt.tight_layout(rect=[0, 0.05, 1, 0.93])
        plt.savefig(os.path.join(output_dir, f"{i}_{domain_id}.png"), dpi=300)
        plt.close(fig)
        
        print(f"Generated visualization for protein {i+1}: {domain_id}")



# Usage example:
#visualize_cath_predictions(df, "output_visualizations", num_visualizations=150)

In [18]:
import os
import py3Dmol
import matplotlib.colors as mcolors


# ---------- Setup ----------
# Create output directory if it doesn't exist
output_dir = "3d_visualizations"
os.makedirs(output_dir, exist_ok=True)

#100, 3
# Select domain index
domain_index = 15

# Get protein and domain info
protein_id = df["protein_id"][domain_index]
domain_start = df["domain_start"][domain_index]
domain_end = df["domain_end"][domain_index]

real_domain_start = df["real_domain_start"][domain_index]
real_domain_end = df["real_domain_end"][domain_index]

# Get colors for visualization
color_actual = mcolors.to_hex(get_cath_color(df["cath"][domain_index])[:3])
color_predicted = mcolors.to_hex(get_cath_color(df["cath_prediction"][domain_index])[:3])

# ---------- View 1: Predicted Domain ----------
view_predicted = py3Dmol.view(query='pdb:' + protein_id)
view_predicted.setStyle({'cartoon': {'color': '#E0E0E0'}})
view_predicted.addStyle({'resi': f"{domain_start}-{domain_end}"}, {'cartoon': {'color': color_predicted}})
view_predicted.zoomTo()

# Show in notebook (optional)
view_predicted.show()

# ---------- View 2: Actual Domain ----------
view_actual = py3Dmol.view(query='pdb:' + protein_id)
view_actual.setStyle({'cartoon': {'color': '#E0E0E0'}})
view_actual.addStyle({'resi': f"{real_domain_start}-{real_domain_end}"}, {'cartoon': {'color': color_actual}})
view_actual.zoomTo()

# Show in notebook (optional)
view_actual.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.